Este tutorial presenta un ejemplo de pronóstico estacional de caudales utilizando el método de los ***análogos históricos***. En resumen, este método asume que sequéncias históricas de los caudales y que son similares a las observaciones recientes, en los meses correspondientes del calendário, serán representativas de los caudales futuros.

Los scripts abajo son basados en la metodología presentada en:


*   *Svensson, C. (2016). Seasonal river flow forecasts for the United Kingdom using persistence and historical analogues. Hydrological Sciences Journal, 61(1), 19-35.*

El ejemplo de cálculo elaborado en este tutorial se refiere al método "*Weigthed mean analogue forecast*". En este caso, el pronóstico resulta de una media ponderada de los análogos, donde el peso está relacionado al error (RMSE) de las anomalías correspondientes a estos análogos respecto a las anomalias observadas en los meses recientes.

---

# **Clonando el repositório del HydroSOS**


Clonaremos el repositório del WMO HydroSOS hospedado en Github para colectar una serie temporal ejemplo de caudales diarios.

Obs: Para utilizar el comando git clone en alguna plataforma de ejecución de notebooks/scripts python en la máquina local (e.g., ***Jupyter***, ***Visual Studio Code***), hay que certificarse de que el Git se encuentra instalado en su ordenador. En caso contrário, el git se puede descargar en (https://git-scm.com/downloads)

In [ ]:
!git clone https://github.com/wmo-im/HydroSOS/

---

# **Importando librerías python**

Importaremos las librerías necesarias para la realización del tutorial:

In [ ]:
import os
import pandas as pd
import numpy as np

from pathlib import Path
from datetime import datetime
from matplotlib import pyplot as plt

---

# **Creando carpetas**

Crearemos automaticamente una carpeta output en el diretorio raiz donde vamos exportar los resultados:

In [ ]:
# Define folder path (e.g., in the current working directory)
folder_output = 'output'

# Create the folder if it doesn't exist
os.makedirs(folder_output, exist_ok=True)

---

# **1. Definición de los parámetros**

Aquí definiremos los parámetros para generación del pronóstico basado en los análogos históricos:

*   `File  path`: Camino donde se encuentra el archivo .csv con los caudales diarios
*   `fcst_range`: Alcance (en meses) del pronóstico pretendido
*   `N_analogs`: Número máximo de análogos permitidos
*   `analog_length`: Duración (en meses) de la sequéncia histórica para comparar las anomalías
*   `apply_transformation`: Indica si queremos aplicar transformación logaritmica a los caudales mensuales (True/False)
*   `min_valid_frac`: Fracción mínima de datos diários válidos en un mes para converterlos a promedio mensual

*   `clim_startYear`: Año inicial del período de referéncia para el cálculo de las anomalias
*   `clim_endYear`: Año final del período de referéncia para el cálculo de las anomalias


In [ ]:
file_path = './HydroSOS/example_data/forecast/input/obs_dir/obs_1.csv' # Path to your CSV file
fcst_range = 3                    # Forecast range (months ahead)
N_analogs = 5                     # Number of analogs from the past (ensemble members)
analog_length = 6                 # Lenght (in months) of the historical sequences to compare anomalies

apply_transformation = True       # Set to True to apply log transformation
min_valid_frac = 0.5              # Minimum fraction of valid daily data per month

clim_startYear = 1991             # Start year to compute anomalies (mean, standard deviation)
clim_endYear = 2020               # End year to compute anomalies

---

# **2. Lectura de los datos**

2.1. Haremos la lectura del archivo .csv con los datos de caudales diários y los convertiremos a mensuales. El archivo .csv debe tener 2 columnas (una con las fechas, y otra con los valores de caudales), además de:

1.   El titulo de la columna con las fechas definido como '*Date*' (sin aspas)
2.   Las fechas en formato **YYYY-MM-DD**

In [ ]:
# Read and preprocess data
df = pd.read_csv(file_path)
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')

df.set_index('Date', inplace=True)
df.columns = ['Discharge'] # ensure the name of the discharge column in csv

# Convert to monthly data
df_month = df.resample('MS').mean()

2.2. Plotando el gráfico de los caudales:

In [ ]:
#@title Grafico
# Plot on screen (subplot for daily and monthly data)
from matplotlib import pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes = axes.flatten()

df.plot(ax=axes[0], color='k', linewidth=0.5, title='Diario')
df_month.plot(ax=axes[1], color='k', linewidth=0.5, title='Mensual')

for ax in axes:
  ax.set_ylabel('Caudal (m³/s)')
  ax.legend(['Observado'])

plt.tight_layout()
plt.show()

---

# **3. Pré-procesamiento de los datos mensuales**

3.1. Removemos los meses con insuficiencia de datos diarios de acuerdo con el mínimo definido en la sección de parámetros:



In [ ]:
# Resample daily to monthly step and compute de fraction of dates in each month without missing data
valid_fraction = df.resample('MS').apply(lambda x: x.count() / x.size)

# Replace monthly values to NaN if % of valid < minimum
df_month.mask(valid_fraction < min_valid_frac, np.nan, inplace=True)

3.2. Aplicando transformación logarítmica $y = log(Q)$ sobre la série de caudales mensuales (si la opción ha sido activada por el usuario en la definición de los parámetros). Este proceso debería dejar la distribuición de los caudales más similar a la normal para el cálculo de las anomalías mensuales estandarizadas:

In [ ]:
# Apply log transformation if selected
if apply_transformation:
    if (df_month <= 0).sum().values > 0:
        raise ValueError("Log transformation requires all values to be > 0.")
    df_month_transformed = np.log(df_month)
else:
    df_month_transformed = df_month.copy()

3.3. Podemos verificar el histograma de los caudales para ver como está la distribuición:

In [ ]:
# Plot the discharge distribution in a histogram
df_month_transformed.hist()

---

# **4. Cálculo de anomalías estándares**

4.1. Cálculo del promedio $ (\overline{x}_m) $ y desvio estándar $ (\sigma_m) $ referente a cada mes del calendario. Las estadísticas son calculadas para el período climatológico de referéncia:

In [ ]:
# Compute monthly climatology (mean and std for each calendar month)
dt_start = datetime(clim_startYear, 1, 1)
dt_end = datetime(clim_endYear, 12, 31)

# Select period to compute anomalies
monthly_transformed_clim = df_month_transformed[(df_month_transformed.index >= dt_start) &
                                               (df_month_transformed.index <= dt_end)]

# Compute mean and standard deviation
month_nums = monthly_transformed_clim.index.month
clim_mean = monthly_transformed_clim.groupby(month_nums).mean()
clim_std = monthly_transformed_clim.groupby(month_nums).std()

4.2. Plotando los valores de promedio y desvio estándar:

In [ ]:
# For visualization purposes only
conc_data = pd.concat([clim_mean, clim_std], axis=1)
conc_data.columns = ['Promedio', 'Desvio std']
conc_data.index.name = 'Mes'
display(conc_data)

4.3. Cálculo de anomalias estandarizadas para cada valor de la serie mensual. El cálculo utiliza el promedio $ (\overline{x_m}) $ y desvio estándar $ (\sigma_m) $ del mes correspondiente al valor observado, conforme la siguiente equación:  

\begin{align}
  a_{t,m} = \frac{q_{t,m}-\overline{x_m}}{\sigma_m}
\end{align}

Donde:

*   $ a_{t,m} $ es la anomalía para el intervalo de tiempo $t$ y mes $m$;
*   $ q_{t,m} $ es caudal (transformado) para el intervalo de tiempo $t$ y mes $m$.

In [ ]:
# Create a copy of the data series and update it to anomalies
df_anomalies = df_month_transformed.copy()

# Compute anomalies for each month separately
for i in range(1, 13):
  idx = (df_anomalies.index.month == i)
  df_anomalies.loc[idx] = (df_month_transformed.loc[idx] - clim_mean.loc[i].values) / clim_std.loc[i].values


4.4. Identificamos las anomalías para los meses recientes considerando una secuéncia de duración definida (`analog_length`):

In [ ]:
# Get anomalies for the recent months
recent_anomalies = df_anomalies[-analog_length:]

4.5. Plotando el gráfico de las anomalias en los últimos meses:

In [ ]:
#@title Grafico
# Plot recent anomalies
recent_anomalies.plot(color='b', marker='o')
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.ylabel('Anomalía estándarizada')
plt.title(f'Anomalías observadas en los últimos {analog_length} meses')
plt.show()

---

# **5. Identificación de los análogos potenciales**

5.1. En esta etapa identificaremos los análogos candidatos (i.e., potenciales) a representar nuestro pronóstico. Por general, la secuéncia de datos en cada año del passado, para las mismas fechas de las anomalías observadas para los últimos meses, se refiere a un análogo potencial, desde que no contenga datos inválidos. Primeramente debemos buscar, en el histórico disponible, todas las fechas correspondientes al mes del calendario en el que inician los análogos:

In [ ]:
# Get the starting month and year of the period corresponding to recent anomalies
analog_start_month = recent_anomalies.index[-analog_length].month
analog_start_year = recent_anomalies.index[-analog_length].year

# Find dates of all calendar months (of previous years) = analog start month and < analog start year
start_indices = df_anomalies.index[(df_anomalies.index.month == analog_start_month) &
                                  (df_anomalies.index.year < analog_start_year)]

# Print starting dates of potential historical analogues
print('Fechas de início para los análogos potenciales:')
display(start_indices)

5.2. Podemos construir una tabla con las fechas para toda la secuencia de interés (considerando la duración $ D_{ana} $ = `analog_length`) y cada análogo potencial:


In [ ]:
# Get the sequences of potential historical analog dates
analog_indices = {
    idx.year: pd.date_range(start=idx, periods=analog_length, freq='MS')
    for idx in start_indices}

# Generate dataframe
df_analog_dates = pd.DataFrame(analog_indices, index=np.arange(1, analog_length+1))

print('Fechas de análogos potenciales:')
display(df_analog_dates)

5.3. Mapeamos los valores de nuestra serie mensual de anomalías para las fechas anteriormente identificadas en cada secuéncia de análogos:

In [ ]:
# Mapping anomalies from time series to their corresponding dates in table
df_analog_anomalies = df_analog_dates.map(lambda t: df_anomalies.loc[t, 'Discharge'] if t in df_anomalies.index else np.nan)

print('Anomalías de cada análogo potencial:')
display(df_analog_anomalies)

5.4. El mismo procedimiento se hace para las fechas y anomalías de pronóstico, que en la práctica se refieren a la extensión de las secuéncias de análogos potenciales para fechas posteriores $ (D_{ana}+m) $, donde $m$ es un índice de mes que va de 1 a `fcst_range`:

In [ ]:
# Select dates corresponding to the last row of potential analog sequences
start_fcst_indices = df_analog_dates.iloc[-1]

# For each column, generate forecast ranges starting from the value in the last row (w/ one month offset)
# This is the same as (Dana + m)
fcst_indices = {
    col: pd.date_range(start=start_fcst_indices[col] + pd.DateOffset(months=1), periods=fcst_range, freq='MS')  # MS = Month Start
    for col in df_analog_dates.columns}

# Create dataframe of historic forecast dates
df_fcst_analog_dates = pd.DataFrame(fcst_indices, index=np.arange(1, fcst_range+1))
# Map anomalies to their corresponding dates
df_fcst_analog_anom = df_fcst_analog_dates.map(lambda t: df_anomalies.loc[t, 'Discharge'] if t in df_anomalies.index else np.nan)

print('Fechas de pronóstico relacionadas a los análogos potenciales:')
display(df_fcst_analog_dates)
print('Anomalías de pronóstico de cada análogo potencial:')
display(df_fcst_analog_anom)

5.5. Cuando hay datos faltantes en la secuéncia del análogo potencial o del pronóstico correspondiente, el analogo deberá ser excluído de la lista:

In [ ]:
# Check if there are missing months in analogs or analog forecasts
analog_missing = np.isnan(df_analog_anomalies).sum(axis=0)
fcst_missing = np.isnan(df_fcst_analog_anom).sum(axis=0)

# If there are years with missing data either in analog or its corresponding forecast, drop them
missing_years = analog_missing + fcst_missing
if missing_years.sum() > 0:
  idx = missing_years[missing_years > 0].index
  df_analog_anomalies = df_analog_anomalies.drop(idx, axis=1)
  df_fcst_analog_anom = df_fcst_analog_anom.drop(idx, axis=1)

display(df_fcst_analog_anom)


---


# **6. Evaluación de los análogos potenciales**

6.1. A continuación, iremos evaluar la adecuación de cada análogo como información futura. Para eso, calculamos el RMSE entre las anomalías recientes y las anomalías de cada secuéncia histórica de análogos potenciales:

\begin{align}
  RMSE = \sqrt{\frac{1}{D_{ana}}\sum_{k=1}^{D_{ana}}{(a_p(k)-a_r(k))^2}}
\end{align}

Donde:

*   $ D_{ana} $ es la duración del análogo;
*   $ a_p(k) $ es la anomalía de caudal para cada mes *k* en el análogo potencial;
*   $ a_r(k) $ es la anomalía correspondiente en el pasado reciente.



In [ ]:
rmse_values = []
# Year loop
for col in df_analog_anomalies.columns:

  # Get historical sequence
  historical_sequence = df_analog_anomalies[col].values
  current_sequence = recent_anomalies.values.flatten()

  # Compute RMSE
  rmse = np.sqrt(np.mean((current_sequence - historical_sequence)**2.0))

  # Include the computed RMSE in list
  rmse_values.append(rmse)

df_rmse = pd.DataFrame(rmse_values, index=df_analog_anomalies.columns, columns=['RMSE'])

6.2. Plotando el gráfico de las diferencias (RMSE) entre las anomalías de la secuéncia reciente y de cada análogo potencial (año del histórico).


In [ ]:
#@title Grafico
# Plot RMSE of analogs from past years
df_rmse['RMSE'].plot(figsize=(12,4))
plt.ylabel('RMSE (-)')
plt.title('RMSE entre las anomalías recientes y las anomalías de cada análogo potencial (año de início)')
plt.show()

---

# **7. Selección final de los análogos históricos**

7.1. Seleccionaremos los `N_analogs` con las menores diferencias (RMSE) respecto a la secuéncia de anomalía reciente:

In [ ]:
# Sort dataframe by rmse values (ascending order)
df_rmse_sorted = df_rmse.sort_values(by='RMSE', ascending=True)

# Select only the first N analogs (with lower RMSE)
df_rmse_sorted = df_rmse_sorted.iloc[0:N_analogs]

print('Análogos seleccionados:')
display(df_rmse_sorted.head(N_analogs))

7.2. Trás la selección de los análogos históricos de menor RMSE, calculamos los pesos según la siguiente equación:

\begin{align}
  w(b) = \frac{1}{RMSE(b)} / \sum_{b=1}^{N_{ana}} \frac{1}{RMSE(b)}
\end{align}

Donde:

*   $ w(b) $ es el peso del análogo $b$;
*   $ N_{ana} $ es el número de análogos seleccionados.

In [ ]:
# Compute weights for each analog based on its (inverse) RMSE
rmse_sum = (1.0 / df_rmse_sorted['RMSE']).sum()
df_rmse_sorted['weight'] = (1.0 / df_rmse_sorted['RMSE']) / rmse_sum

print('Análogos seleccionados y pesos:')
display(df_rmse_sorted.head(N_analogs))

print(f'Suma de los pesos: {df_rmse_sorted["weight"].sum(axis=0)}')

---


# **8. Determinación del pronóstico basado en análogos**

8.1. Buscaremos las series de anomalías de “pronóstico” que corresponden a los análogos seleccionados (de menor RMSE):  

In [ ]:
# Retrieving only the analog forecast sequences corresponding to years with lowest RMSE
fcst_selected = df_fcst_analog_anom[df_rmse_sorted.index]

display(fcst_selected)

8.2. Cálculo de la anomalía promedia pronosticada según la ponderación por el RMSE, utilizando los análogos seleccionados:

\begin{align}
  a_f(m) = \sum_{b=1}^{N_{ana}}{w(b)a_{p,b}(D_{ana}+m)}
\end{align}

Donde:

*   $ a_f(m) $ es la anomalía de pronóstico ponderada, para el mes $m$;
*   $ a_{p,b}(D_{ana}+m) $ es la anomalía de pronóstico correspondiente al análogo $b$, para el mes $m$;


In [ ]:
# Get weights
weights = df_rmse_sorted['weight']
# Multiply matrices (analogues x weights) and sum values
df_weighted_anomaly = fcst_selected.mul(weights).sum(axis=1)

print('Anomalias de pronóstico (ponderadas por RMSE):')
display(df_weighted_anomaly)

8.3. Identificamos la fecha más reciente:

In [ ]:
# Check standardized anomaly for last month
most_recent_date = recent_anomalies.index[-1]

print(f'Mes más reciente de observaciones: {most_recent_date.strftime("%Y-%m")}')

8.4. Identificamos las fechas de los *N* meses del pronóstico y calculamos los caudales a partir de las anomalías ponderadas pronosticadas, a través de la equación:  

\begin{align}
  q_m = \overline{x_m} + (a_m \times \sigma_m)
\end{align}

Donde:

*   $ a_{m} $ es la anomalía prevista en el mes $m$;

In [ ]:
# Generate forecast dates
forecast_dates = pd.date_range(start=most_recent_date + pd.offsets.MonthEnd(1),
                               periods=fcst_range,
                               freq='MS')

# Get forecast months
forecast_month_nums = forecast_dates.month

# Get the climatological mean and standard deviation for these months
clim_mean_fcst = clim_mean.loc[forecast_month_nums].values.flatten()
clim_std_fcst = clim_std.loc[forecast_month_nums].values.flatten()

# Reconstruct forecast values from anomalies, based on climatological means and standard deviations
forecast_transformed = clim_mean_fcst + (df_weighted_anomaly.values * clim_std_fcst)

# Construct dataframe
df_forecast_transformed = pd.DataFrame(forecast_transformed, index=forecast_dates, columns=['Discharge'])
df_forecast_transformed.index.name = 'Date'

print('Caudales pronosticados (transformados):')
display(df_forecast_transformed)

8.5. Para fines de graficar los resultados, también podemos obtener los caudales de los análogos para todo el rango de fechas de interés, es decir, para $D_{ana}$ + `fcst_range`:

In [ ]:
# Concatenating dates: selected analogs and corresponding "forecasts"
df_analog_dates_full = pd.concat([df_analog_dates, df_fcst_analog_dates], axis=0)

# Mapping (transfromed) discharges from time series to their corresponding dates in table
df_analog_full = df_analog_dates_full.map(lambda t: df_month_transformed.loc[t, 'Discharge'] if t in df_month_transformed.index else np.nan)

# Get only the N_analogs with lowest RMSE
df_analog_full = df_analog_full[df_rmse_sorted.index]

analog_dates = pd.date_range(start=recent_anomalies.index[0],
                               periods= (analog_length + fcst_range),
                               freq='MS')

df_analog_full.index = analog_dates
df_analog_full.index.name = 'Date'

print('Caudales para análogos seleccionados (transformados):')
display(df_analog_full)

8.6. Transformación de los caudales de vuelta al espacio real a través de $ ( Q = e^y ) $ (en caso que hayan sido transformados anteriormente):

In [ ]:
# Back-transform if log transformation was applied
if apply_transformation:
    df_forecast = np.exp(df_forecast_transformed)
    df_analogs = np.exp(df_analog_full)
else:
    df_forecast = df_forecast_transformed.copy()
    df_analogs = df_analog_full.copy()

df_forecast[df_forecast < 0.0] = 0.0

8.7. Plotando el grafico final con los pronósticos:

In [ ]:
#@title Grafico
# Plot on screen
fig, ax = plt.subplots(figsize=(15, 4))

df_month.iloc[-36:].plot(ax=ax, color='k')
df_analogs.plot(ax=ax, color='gray', alpha=0.5)
df_forecast.plot(ax=ax, color='r', linewidth=2.5, linestyle='--')

# This part is only for visualization purposes
# Adjust legend labels
h, lbl = ax.get_legend_handles_labels()
lbl[0] = 'Observado'
lbl[-1] = 'Pronóstico'
ax.legend(h, lbl)

ax.set_ylabel('Caudal (m³/s)')
ax.set_title('Análogos historicos')

plt.show()

---

# **9. Exportando los pronósticos generados**

Por último, podemos exportar nuestras previsiones de caudal a la carpeta `output` indicada:

In [ ]:
# Save to CSV
file_name = Path(file_path).stem
df_forecast.to_csv(f'{folder_output}/analog_forecast_{file_name}.csv')